# 🏠 DOMIAN — Entrenamiento Local (sin GPU)
## Adaptive Classifier · 3 clases · CPU

**Clases:**
- `absent` → habitación vacía
- `present_still` → persona quieta  
- `present_moving` → persona moviéndose

**Requisitos:**
```bash
pip install numpy scikit-learn
```

**Compatibilidad:** Mac Intel, Mac M1/M2, Windows, Linux — sin GPU requerida

---
**Grabaciones deben estar en:**
```
~/Documents/RuView/v2/data/recordings/
```
Con prefijos: `train_absent_`, `train_present_still_`, `train_present_moving_`

## Paso 1 — Instalar dependencias

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'numpy', 'scikit-learn', '-q'])
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import json, os
from pathlib import Path
print('✅ Dependencias instaladas')
print(f'   numpy {np.__version__}')

## Paso 2 — Configurar rutas

In [ ]:
import os

# Ruta de grabaciones
RECORDINGS_DIR = os.path.expanduser('~/Documents/RuView/v2/data/recordings')
MODEL_PATH     = os.path.expanduser('~/Documents/RuView/v2/data/adaptive_model.json')

CLASE_NOMBRES = ['absent', 'present_still', 'present_moving']
LABEL_MAP = {
    'train_absent':         0,
    'train_present_still':  1,
    'train_present_moving': 2,
}

# Verificar directorio
if not os.path.exists(RECORDINGS_DIR):
    print(f'❌ No encontrado: {RECORDINGS_DIR}')
else:
    archivos = sorted(os.listdir(RECORDINGS_DIR))
    print(f'✅ Directorio: {RECORDINGS_DIR}')
    print(f'   Archivos encontrados: {len(archivos)}')
    for f in archivos:
        size = os.path.getsize(f'{RECORDINGS_DIR}/{f}') / 1024 / 1024
        label = next((LABEL_MAP[p] for p in LABEL_MAP if f.startswith(p)), None)
        clase = CLASE_NOMBRES[label] if label is not None else '⚠️ sin prefijo'
        print(f'   {f} — {size:.1f} MB → {clase}')

## Paso 3 — Cargar datos (features 15-dim)

In [ ]:
def extraer_features_15(obj):
    """Extrae 15 features por nodo — compatible con N_FEATURES=15 del servidor RuView"""
    resultados = []
    for nf in obj.get('node_features', []):
        f   = nf.get('features', {})
        clf = nf.get('classification', {})
        row = [
            float(f.get('mean_rssi', 0)),
            float(f.get('variance', 0)),
            float(f.get('motion_band_power', 0)),
            float(f.get('breathing_band_power', 0)),
            float(f.get('dominant_freq_hz', 0)),
            float(f.get('change_points', 0)),
            float(f.get('spectral_power', 0)),
            float(nf.get('rssi_dbm', 0)),
            float(nf.get('last_seen_ms', 0)),
            float(nf.get('frame_rate_hz', 0)),
            1.0 if nf.get('stale') else 0.0,
            1.0 if clf.get('presence') else 0.0,
            float(clf.get('confidence', 0)),
            float(nf.get('node_id', 0)),
            0.0  # padding
        ]
        resultados.append(row)
    return resultados

def cargar_grabacion(path, label):
    frames = []
    with open(path, 'r') as f:
        for linea in f:
            try:
                obj = json.loads(linea.strip())
                for row in extraer_features_15(obj):
                    frames.append((np.array(row, dtype=np.float32), label))
            except:
                continue
    return frames

# Cargar todos los archivos con prefijo correcto
datos = []
archivos = sorted(os.listdir(RECORDINGS_DIR))

print('Cargando grabaciones...')
for archivo in archivos:
    label = None
    for prefijo, lbl in LABEL_MAP.items():
        if archivo.startswith(prefijo):
            label = lbl
            break
    if label is None:
        print(f'  ⚠️  {archivo} — ignorado (sin prefijo train_*)')
        continue
    path   = f'{RECORDINGS_DIR}/{archivo}'
    frames = cargar_grabacion(path, label)
    print(f'  ✅ {archivo}: {len(frames):,} frames → {CLASE_NOMBRES[label]}')
    datos += frames

if not datos:
    print('\n❌ Sin datos. Verifica que los archivos tengan prefijos train_absent_, train_present_still_ o train_present_moving_')
else:
    print(f'\nTotal: {len(datos):,} frames')
    for i, nombre in enumerate(CLASE_NOMBRES):
        count = sum(1 for _, l in datos if l == i)
        pct   = 100 * count / len(datos) if datos else 0
        print(f'  {nombre:20}: {count:,} ({pct:.1f}%)')

## Paso 4 — Entrenar con Logistic Regression

Regresión logística multiclase — mismo algoritmo del servidor RuView, sin GPU

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import time

X = np.array([f for f, _ in datos])
y = np.array([l for _, l in datos])

# Split estratificado
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Normalizar
scaler = StandardScaler()
X_train_n = scaler.fit_transform(X_train)
X_val_n   = scaler.transform(X_val)

mean15 = scaler.mean_
std15  = scaler.scale_

print(f'Train: {len(X_train):,} frames')
print(f'Val:   {len(X_val):,} frames')
print('\nEntrenando...')

t0  = time.time()
clf = LogisticRegression(
    max_iter=1000,
    C=1.0,
    multi_class='multinomial',
    solver='lbfgs',
    random_state=42,
    n_jobs=-1
)
clf.fit(X_train_n, y_train)
elapsed = time.time() - t0

acc = clf.score(X_val_n, y_val)
print(f'\n✅ Entrenamiento completado en {elapsed:.1f}s')
print(f'   Accuracy: {acc:.4f} ({acc*100:.1f}%)')
print()
print(classification_report(y_val, clf.predict(X_val_n), target_names=CLASE_NOMBRES))

## Paso 5 — Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np

cm = confusion_matrix(y_val, clf.predict(X_val_n))

print('Confusion Matrix (filas=real, columnas=predicho):')
print(f'{"": <20}', end='')
for c in CLASE_NOMBRES:
    print(f'{c[:12]:>14}', end='')
print()
print('-' * 62)
for i, row in enumerate(cm):
    print(f'{CLASE_NOMBRES[i]:20}', end='')
    for val in row:
        print(f'{val:>14,}', end='')
    print()

print()
print('Precisión por clase:')
for i, nombre in enumerate(CLASE_NOMBRES):
    correctos = cm[i][i]
    total     = cm[i].sum()
    print(f'  {nombre:20}: {correctos:,}/{total:,} = {100*correctos/total:.1f}%')

## Paso 6 — Exportar en formato RuView

Genera `adaptive_model.json` compatible con el servidor Rust

In [ ]:
import json

# Los pesos de LogisticRegression ya son [n_classes x n_features]
# El servidor espera [n_classes x (n_features + 1)] — última columna = bias
weights = []
for i in range(len(CLASE_NOMBRES)):
    row = clf.coef_[i].tolist() + [float(clf.intercept_[i])]
    weights.append(row)

print(f'Weights: {len(weights)} clases x {len(weights[0])} (features + bias)')

# Stats por clase
class_stats = []
for label_id, nombre in enumerate(CLASE_NOMBRES):
    mask  = y == label_id
    X_cls = X[mask]
    if len(X_cls) == 0:
        print(f'  ⚠️  Sin datos para clase {nombre}')
        class_stats.append({'label': nombre, 'count': 0, 'mean': [0]*15, 'stddev': [1]*15})
        continue
    class_stats.append({
        'label':  nombre,
        'count':  int(mask.sum()),
        'mean':   X_cls.mean(axis=0).tolist(),
        'stddev': X_cls.std(axis=0).tolist()
    })
    print(f'  {nombre:20}: {mask.sum():,} frames')

modelo_ruview = {
    'class_stats':       class_stats,
    'weights':           weights,
    'global_mean':       mean15.tolist(),
    'global_std':        std15.tolist(),
    'trained_frames':    len(X),
    'training_accuracy': float(acc),
    'version':           1,
    'class_names':       CLASE_NOMBRES
}

# Guardar
with open(MODEL_PATH, 'w') as f:
    json.dump(modelo_ruview, f, indent=2)

print(f'\n✅ Modelo guardado en:')
print(f'   {MODEL_PATH}')
print(f'\n   Accuracy: {acc:.4f} ({acc*100:.1f}%)')
print(f'   Clases:   {CLASE_NOMBRES}')
print(f'   Frames:   {len(X):,}')

## Paso 7 — Reiniciar servidor y verificar

El modelo se cargará automáticamente al reiniciar el servidor

In [ ]:
print('Pasos para activar el nuevo modelo:')
print()
print('1. Detén el servidor con Ctrl+C')
print()
print('2. Reinicia el servidor:')
print('   cd ~/Documents/RuView/v2')
print('   cargo run -p wifi-densepose-sensing-server -- \\')
print('     --source esp32 --bind-addr 0.0.0.0 \\')
print('     --udp-port 5005 --http-port 8000 \\')
print('     --allowed-host 192.168.0.8 --allowed-host localhost \\')
print('     --node-positions "0,0,1.5;0,3.5,1.5;5.7,3.5,1.5;5.7,0,1.5" \\')
print('     --load-rvf ~/Documents/RuView/_Domian/demo/models/laoracion.rvf')
print()
print('3. Verifica en los logs:')
print(f'   Loaded adaptive classifier: {len(X):,} frames, {acc*100:.1f}% accuracy')
print()
print('4. Prueba rápida:')
print('   curl http://192.168.0.8:8000/api/v1/adaptive/status')

## Paso 8 — Prueba de inferencia local

In [ ]:
import numpy as np

def predecir(features_raw):
    x = (np.array(features_raw) - mean15) / std15
    probs = clf.predict_proba([x])[0]
    clase = CLASE_NOMBRES[probs.argmax()]
    return clase, probs

# Frames de prueba típicos
tests = [
    ('habitación vacía',   [0, 2.1, 1.5, 0.8, 0.3, 0, 5.2, -75, 50, 10, 0, 0, 0.1, 2, 0]),
    ('persona quieta',     [-65, 19.8, 33.9, 28.1, 1.2, 5, 118.7, -65, 30, 10, 0, 1, 0.33, 2, 0]),
    ('persona moviéndose', [-62, 35.2, 58.4, 45.3, 2.1, 12, 145.2, -62, 25, 10, 0, 1, 0.85, 2, 0]),
]

print('Prueba de inferencia:')
print('-' * 55)
for nombre, features in tests:
    clase, probs = predecir(features)
    print(f'{nombre:25} → {clase:20} ({probs.max():.2f})')
    for i, c in enumerate(CLASE_NOMBRES):
        bar = '█' * int(probs[i] * 20)
        print(f'   {c:20} {bar:<20} {probs[i]:.3f}')
    print()